## Импорт библиотек и настройки

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import zipfile
from pathlib import Path
from typing import Tuple, List, Dict, Any
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import LabelEncoder
from sklearn.base import clone
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)

RANDOM_STATE = 42
ROOT = Path.cwd()
DATA_DIR = ROOT / "data" if (ROOT / "data").exists() else ROOT

print("✅ Окружение настроено")

✅ Окружение настроено


In [5]:
print(f"Root: {ROOT}")

Root: /home/yamshchikov/ML_practice/Torchvision_core_project/Shift_ML_Tech_Task_2/notebooks


## Загрузка данных

In [6]:
print("📥 Загрузка данных...")
train = pd.read_csv(DATA_DIR / "../data/train.csv")
test = pd.read_csv(DATA_DIR / "../data/test.csv")
bureau = pd.read_csv(DATA_DIR / "../data/bureau.csv")
transactions = pd.read_csv(DATA_DIR / "../data/transactions.csv")
previous_loans = pd.read_csv(DATA_DIR / "../data/previous_loans.csv")

print(f"Train: {train.shape} | Test: {test.shape}")
print(f"Bureau: {bureau.shape} | Transactions: {transactions.shape} | Previous Loans: {previous_loans.shape}")

# Базовый EDA: баланс классов
target_rate = train['target'].mean()
print(f"📊 Доля дефолтов (target=1) в train: {target_rate:.2%}")

📥 Загрузка данных...
Train: (6488, 25) | Test: (2520, 24)
Bureau: (24689, 8) | Transactions: (353300, 6) | Previous Loans: (12762, 7)
📊 Доля дефолтов (target=1) в train: 34.22%


## Агрегация реляционных данных (1-ко-многим)

In [7]:
def aggregate_relational_data(train_df, test_df, bureau_df, prev_loans_df, tx_df):
    print("🔄 Начало агрегации реляционных данных...")
    
    # 1. Bureau
    bureau_agg = bureau_df.groupby('client_id').agg(
        bureau_accounts_count=('bureau_account_id', 'count'),
        total_credit_limit=('credit_limit', 'sum'),
        total_current_balance=('current_balance', 'sum'),
        max_dpd_last_12m=('max_dpd_last_12m', 'max'),
        active_accounts_count=('bureau_status', lambda x: (x == 'active').sum()),
    ).reset_index()
    bureau_agg['credit_utilization_ratio'] = bureau_agg['total_current_balance'] / (bureau_agg['total_credit_limit'] + 1e-5)
    
    # 2. Previous Loans
    prev_agg = prev_loans_df.groupby('client_id').agg(
        prev_loans_count=('previous_loan_id', 'count'),
        total_prev_amount=('previous_amount', 'sum'),
        max_overdue_days=('max_overdue_days', 'max'),
        was_overdue_count=('was_overdue', 'sum'),
    ).reset_index()
    prev_agg['was_overdue_ratio'] = prev_agg['was_overdue_count'] / (prev_agg['prev_loans_count'] + 1e-5)
    
    # 3. Transactions (агрегируем по client_id для исторического контекста)
    tx_df['transaction_date'] = pd.to_datetime(tx_df['transaction_date'], errors='coerce')
    ref_date = pd.to_datetime('2025-12-31')
    tx_df['days_since_tx'] = (ref_date - tx_df['transaction_date']).dt.days
    
    tx_agg = tx_df.groupby('client_id').agg(
        tx_count=('transaction_date', 'count'),
        net_tx_amount=('amount', 'sum'),
        total_tx_turnover=('amount', lambda x: x.abs().sum()),
        high_risk_tx_count=('merchant_risk_level', lambda x: (x >= 4).sum()),
        days_since_last_tx=('days_since_tx', 'min')
    ).reset_index()
    tx_agg['high_risk_ratio'] = tx_agg['high_risk_tx_count'] / (tx_agg['tx_count'] + 1e-5)
    
    # 4. Слияние
    result_dfs = []
    for df in [train_df, test_df]:
        df_merged = df.copy()
        df_merged = df_merged.merge(bureau_agg, on='client_id', how='left')
        df_merged = df_merged.merge(prev_agg, on='client_id', how='left')
        df_merged = df_merged.merge(tx_agg, on='client_id', how='left')
        result_dfs.append(df_merged)
        
    print("✅ Агрегация завершена.")
    return result_dfs[0], result_dfs[1]

train_agg, test_agg = aggregate_relational_data(train, test, bureau, previous_loans, transactions)

🔄 Начало агрегации реляционных данных...
✅ Агрегация завершена.


## Продвинутый Feature Engineering (Адаптированный)

In [8]:
def create_new_features(df):
    df = df.copy()
    
    # 1. Финансовая нагрузка
    df['monthly_income_safe'] = df['monthly_income'].fillna(df['monthly_income'].median())
    df['loan_to_income_ratio'] = df['loan_amount'] / (df['monthly_income_safe'] * 12 + 1)
    df['estimated_monthly_payment'] = df['loan_amount'] / (df['loan_term_months'].fillna(12) + 1)
    df['payment_to_income_ratio'] = df['estimated_monthly_payment'] / (df['monthly_income_safe'] + 1)
    
    # 2. Риски из бюро и истории
    df['has_overdue_history'] = (df['max_dpd_last_12m'].fillna(0) > 0).astype(int)
    df['bureau_debt_ratio'] = df['total_current_balance'].fillna(0) / (df['total_credit_limit'].fillna(1) + 1e-5)
    
    # 3. Активность транзакций
    df['tx_turnover_to_income'] = df['total_tx_turnover'].fillna(0) / (df['monthly_income_safe'] * 12 + 1)
    
    # 4. Флаги
    df['is_high_risk_employment'] = df['employment_type'].isin(['unemployed', 'contractor', 'self_employed']).astype(int)
    df['is_short_term_loan'] = (df['loan_term_months'].fillna(12) <= 12).astype(int)
    
    return df

train_fe = create_new_features(train_agg)
test_fe = create_new_features(test_agg)
print("✅ Feature Engineering завершен")

✅ Feature Engineering завершен


## Умная предобработка

In [10]:
def advanced_preprocessing(train_df, test_df, target_col='target'):
    print("="*50)
    print("НАЧАЛО ПРЕДОБРАБОТКИ")
    print("="*50)
    
    X_train = train_df.drop(columns=[target_col, 'application_id', 'client_id', 'hash_id'], errors='ignore')
    X_test = test_df.drop(columns=['application_id', 'client_id', 'hash_id'], errors='ignore')
    y_train = train_df[target_col]
    test_ids = test_df['application_id']
    
    # Разделяем на числовые и категориальные
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
    
    print(f"Числовых признаков: {len(num_cols)}, Категориальных: {len(cat_cols)}")
    
    # 1. Обработка числовых признаков (заполняем медианой из TRAIN)
    for col in num_cols:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)
        
    # 2. Обработка категориальных признаков
    for col in cat_cols:
        # Заполняем пропуски специальным значением
        mode_val = X_train[col].mode()[0] if not X_train[col].mode().empty else 'MISSING'
        X_train[col] = X_train[col].fillna('MISSING')
        X_test[col] = X_test[col].fillna('MISSING')
        
        # Label Encoding (обучаем на объединенных данных, чтобы обработать редкие категории в test)
        le = LabelEncoder()
        combined = pd.concat([X_train[col], X_test[col]]).astype(str)
        le.fit(combined)
        X_train[col] = le.transform(X_train[col].astype(str))
        X_test[col] = le.transform(X_test[col].astype(str))
        
    # Выравнивание колонок
    missing_in_test = set(X_train.columns) - set(X_test.columns)
    for col in missing_in_test:
        X_test[col] = 0
        
    extra_in_test = set(X_test.columns) - set(X_train.columns)
    X_test = X_test.drop(columns=list(extra_in_test), errors='ignore')
    X_test = X_test[X_train.columns]
    
    print("✅ Предобработка завершена. Пропусков не осталось.")
    return X_train, X_test, y_train, test_ids

X_train, X_test, y_train, test_ids = advanced_preprocessing(train_fe, test_fe)

НАЧАЛО ПРЕДОБРАБОТКИ
Числовых признаков: 39, Категориальных: 8
✅ Предобработка завершена. Пропусков не осталось.


## Разделение на Train/Validation и Обучение Моделей

In [11]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_STATE, stratify=y_train
)

print(f"Train split: {X_tr.shape}, Validation split: {X_val.shape}")

def train_and_evaluate(X_tr, y_tr, X_val, y_val):
    models = {
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.03, max_depth=8, num_leaves=63,
            subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
            random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, is_unbalance=True
        ),
        'CatBoost': CatBoostClassifier(
            iterations=2000, learning_rate=0.03, depth=8, l2_leaf_reg=5,
            random_state=RANDOM_STATE, verbose=0, auto_class_weights='Balanced'
        )
    }
    
    results = {}
    for name, model in models.items():
        print(f"\n🚀 Обучение {name}...")
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
        
        # Для CatBoost eval_set передается иначе, но predict_proba универсален
        preds = model.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, preds)
        
        results[name] = {'model': model, 'auc': auc, 'preds_val': preds}
        print(f"   ✅ {name} Validation ROC-AUC: {auc:.5f}")
        
    return results

model_results = train_and_evaluate(X_tr, y_tr, X_val, y_val)

Train split: (5190, 47), Validation split: (1298, 47)

🚀 Обучение LightGBM...
   ✅ LightGBM Validation ROC-AUC: 0.98768

🚀 Обучение CatBoost...
   ✅ CatBoost Validation ROC-AUC: 0.98645


## Ансамблирование (Взвешенное усреднение)

In [12]:
print("\n⚖️ Взвешенное усреднение лучших моделей...")

# Собираем предсказания
val_preds_list = []
weights = []
names = []

for name, res in model_results.items():
    val_preds_list.append(res['preds_val'])
    # Вес пропорционален квадрату AUC (усиливает лучшую модель)
    weights.append(res['auc'] ** 2)
    names.append(name)

weights = np.array(weights) / np.sum(weights) # Нормализация
print(f"Веса моделей: {dict(zip(names, np.round(weights, 3)))}")

# Взвешенная сумма
ensemble_val_preds = np.zeros_like(val_preds_list[0])
for i, preds in enumerate(val_preds_list):
    ensemble_val_preds += preds * weights[i]

ensemble_auc = roc_auc_score(y_val, ensemble_val_preds)
print(f"🏆 Ensemble Validation ROC-AUC: {ensemble_auc:.5f}")


⚖️ Взвешенное усреднение лучших моделей...
Веса моделей: {'LightGBM': np.float64(0.501), 'CatBoost': np.float64(0.499)}
🏆 Ensemble Validation ROC-AUC: 0.98743


## Финальное предсказание на Test и создание сабмита

In [ ]:
print("\n🎯 Генерация финальных предсказаний на тесте...")

# Переобучаем лучшую модель на всех данных train
final_model = model_results['LightGBM']['model']
final_model.fit(X_train, y_train)

# Предсказание вероятностей
test_predictions = final_model.predict_proba(X_test)[:, 1]

# Формирование DataFrame
submission = pd.DataFrame({
    "application_id": test_ids,
    "target": test_predictions
})

# Сохранение
submission.to_csv(ROOT / "submission.csv", index=False)
print("✅ Файл submission.csv успешно сохранен!")
display(submission.head())

# Проверка диапазона (ROC-AUC)
print(f"\nСтатистика предсказаний: Min={test_predictions.min():.4f}, Max={test_predictions.max():.4f}, Mean={test_predictions.mean():.4f}")


🎯 Генерация финальных предсказаний на тесте...
✅ Файл submission.csv успешно сохранен!


,application_id,target
0,102531,0.879825
1,107213,0.907664
2,100238,0.064978
3,104918,0.019328
4,106480,0.015277



Статистика предсказаний: Min=0.0000, Max=0.9998, Mean=0.2515


## Генерация requirements.txt и архива submission.zip

In [14]:
print("\n📦 Формирование архива для отправки...")

# 1. Создаем requirements.txt
req_content = """pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.3.0
lightgbm>=4.0.0
catboost>=1.2.0
xgboost>=2.0.0
matplotlib>=3.7.0
seaborn>=0.12.0
"""
with open(ROOT / "requirements.txt", "w") as f:
    f.write(req_content)
print("✅ requirements.txt создан")

# 2. Создаем архив submission.zip
zip_path = ROOT / "submission.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(ROOT / "submission.csv", "submission.csv")
    zipf.write(ROOT / "requirements.txt", "requirements.txt")
    # Если ноутбук называется иначе, поправь имя файла ниже:
    zipf.write(ROOT / "competition.ipynb", "competition.ipynb")

print(f"✅ Архив успешно создан: {zip_path}")
print("🎉 Готово! Можешь загружать submission.zip на платформу.")


📦 Формирование архива для отправки...
✅ requirements.txt создан
✅ Архив успешно создан: /home/yamshchikov/ML_practice/Torchvision_core_project/Shift_ML_Tech_Task_2/notebooks/submission.zip
🎉 Готово! Можешь загружать submission.zip на платформу.
